![logo](https://lever-client-logos.s3.amazonaws.com/2bd4cdf9-37f2-497f-9096-c2793296a75f-1568844229943.png)

# GetAround — Projet Complet : Analyse des Retards & Optimisation des Prix

**Auteur :** Philippe TOSO  
**Date :** Mars 2026  
**Contexte :** Bootcamp Jedha — Bloc 5 — Industrialisation d'un algorithme de Machine Learning

---

## Présentation du Projet

[GetAround](https://www.getaround.com) est le Airbnb de la voiture. Fondée en 2009, la plateforme compte plus de 5 millions d'utilisateurs et environ 20 000 voitures dans le monde.

Ce projet répond à **deux problématiques business** :

### Problème 1 — Retards au retour & Politique de Seuil
Les conducteurs rendent parfois les voitures en retard, ce qui crée des frictions pour le locataire suivant.
- **Seuil :** Quel délai minimum entre deux locations ?
- **Périmètre :** Toutes les voitures, ou seulement Connect ?

### Problème 2 — Optimisation des Prix
Construire un **modèle de régression ML supervisé** pour suggérer le prix de location journalier optimal.

## Structure du Notebook
1. Configuration  2. EDA des Retards  3. Simulation de Seuil  4. EDA des Prix  5. Feature Engineering  6. ML (MLflow)  7. Évaluation

---
## 1. Configuration & Chargement des Données

In [ ]:
# --- Bibliothèques de base ---
import pandas as pd
import numpy as np
import warnings, os, json
warnings.filterwarnings('ignore')

# --- Visualisation ---
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Machine Learning ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- MLflow ---
import mlflow
import mlflow.sklearn

# --- Sérialisation ---
import joblib

print('Toutes les bibliothèques ont été importées avec succès')
print('Version de MLflow :', mlflow.__version__)

In [ ]:
# --- Chargement des deux jeux de données ---
df_delay   = pd.read_excel('../get_around_delay_analysis.xlsx')
df_pricing = pd.read_csv('../get_around_pricing_project.csv')

print('=== Analyse des Retards ===' , df_delay.shape)
display(df_delay.head(3))
print('=== Tarification ===' , df_pricing.shape)
display(df_pricing.head(3))

---
## 2. Partie 1 — EDA de l'Analyse des Retards

### 2.1 Valeurs Manquantes & Qualité des Données

In [ ]:
print('Valeurs manquantes :')
print(df_delay.isnull().sum())
print('\nTypes de checkin :')
print(df_delay['checkin_type'].value_counts())
print('\nÉtats :')
print(df_delay['state'].value_counts())

# Pourcentage de retours en retard (locations terminées uniquement)
ended = df_delay[df_delay['state'] == 'ended']
late    = (ended['delay_at_checkout_in_minutes'] > 0).sum()
on_time = (ended['delay_at_checkout_in_minutes'] <= 0).sum()
print(f'Locations terminées : {len(ended)}')
print(f'  En retard : {late} ({late/len(ended)*100:.1f}%)')
print(f'  À l\'heure : {on_time} ({on_time/len(ended)*100:.1f}%)')

### 2.2 Distribution des Retards au Retour

In [ ]:
ended_with_delay = ended.dropna(subset=['delay_at_checkout_in_minutes'])

# --- Graphique 1 : Histogramme des retards ---
fig1 = px.histogram(
    ended_with_delay, x='delay_at_checkout_in_minutes', color='checkin_type',
    nbins=80, range_x=[-300, 720],
    title='Distribution des retards au retour par type de checkin',
    labels={'delay_at_checkout_in_minutes': 'Retard (min)', 'checkin_type': 'Type de Checkin'},
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'},
    opacity=0.75, barmode='overlay'
)
fig1.add_vline(x=0, line_dash='dash', line_color='black', annotation_text='À l\'heure')
fig1.update_layout(template='plotly_white')
fig1.show()

In [ ]:
# --- Graphique 2 : Box plot des retards par type de checkin ---
fig2 = px.box(
    ended_with_delay, x='checkin_type', y='delay_at_checkout_in_minutes',
    color='checkin_type', points='outliers',
    title='Retard au retour par type de checkin (Box Plot)',
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'}
)
fig2.add_hline(y=0, line_dash='dash', line_color='black')
fig2.update_layout(template='plotly_white', showlegend=False)
fig2.show()
print(ended_with_delay.groupby('checkin_type')['delay_at_checkout_in_minutes'].describe())

In [ ]:
# --- Graphique 3 : Distribution des états par type de checkin ---
state_counts = df_delay.groupby(['checkin_type', 'state']).size().reset_index(name='count')
fig3 = px.bar(state_counts, x='checkin_type', y='count', color='state', barmode='group',
    title='Distribution de l\'état des locations par type de checkin',
    color_discrete_map={'ended': '#00CC96', 'canceled': '#FF6692'})
fig3.update_layout(template='plotly_white')
fig3.show()

### 2.3 Analyse des Locations Consécutives

Seulement **1 841 locations** (~8,6%) ont une réservation précédente — ce sont les cas où un retard au retour impacte réellement le conducteur suivant.

In [ ]:
# --- Filtre : locations consécutives uniquement ---
consecutive = df_delay.dropna(subset=[
    'previous_ended_rental_id', 'time_delta_with_previous_rental_in_minutes'
]).copy()
print(f'Paires consécutives : {len(consecutive)} ({len(consecutive)/len(df_delay)*100:.1f}%)')

# Fusionner le retard de la location précédente
prev_delays = df_delay[['rental_id', 'delay_at_checkout_in_minutes']].rename(
    columns={'rental_id': 'previous_ended_rental_id',
             'delay_at_checkout_in_minutes': 'prev_delay'})
consecutive = consecutive.merge(prev_delays, on='previous_ended_rental_id', how='left')

consecutive['time_delta'] = consecutive['time_delta_with_previous_rental_in_minutes']
consecutive['prev_delay_filled'] = consecutive['prev_delay'].fillna(0)
# Identifier les cas problématiques : retard précédent > delta de temps disponible
consecutive['is_problematic'] = consecutive['prev_delay_filled'] > consecutive['time_delta']
problematic = consecutive[consecutive['is_problematic']]
print(f'Problématiques : {len(problematic)} ({len(problematic)/len(consecutive)*100:.1f}%)')

In [ ]:
# --- Graphique 4 : Distribution du delta de temps entre locations consécutives ---
fig4 = px.histogram(
    consecutive, x='time_delta', color='checkin_type', nbins=50,
    title='Delta de temps entre locations consécutives',
    labels={'time_delta': 'Delta de temps (min)', 'checkin_type': 'Type de Checkin'},
    color_discrete_map={'mobile': '#636EFA', 'connect': '#EF553B'},
    opacity=0.75, barmode='overlay'
)
fig4.update_layout(template='plotly_white')
fig4.show()

---
## 3. Partie 2 — Simulation de Seuil

Nous simulons différents **seuils de délai minimum** (0–720 min) :
- **Problèmes résolus** : cas consécutifs problématiques résolus
- **Chiffre d'affaires impacté** : locations bloquées (delta de temps < seuil)

In [ ]:
# --- Simulation de seuil ---
thresholds = list(range(0, 721, 30))
results = []

for scope in ['all', 'connect']:
    if scope == 'connect':
        consec_scope = consecutive[consecutive['checkin_type'] == 'connect']
        all_scope    = df_delay[df_delay['checkin_type'] == 'connect']
    else:
        consec_scope = consecutive
        all_scope    = df_delay

    prob_scope   = consec_scope[consec_scope['is_problematic']]
    total_prob   = len(prob_scope)
    total_rentals = len(all_scope)

    for t in thresholds:
        solved   = (prob_scope['prev_delay_filled'] <= t).sum()
        affected = (consec_scope['time_delta'] < t).sum()
        results.append({
            'threshold': t, 'scope': scope,
            'pct_problems_solved'  : solved   / total_prob    * 100 if total_prob > 0 else 0,
            'pct_rentals_affected' : affected / total_rentals * 100,
        })

df_sim = pd.DataFrame(results)
print(df_sim[(df_sim['threshold'].isin([0,60,120,180,240])) & (df_sim['scope']=='all')]
      [['threshold','pct_problems_solved','pct_rentals_affected']].to_string(index=False))

In [ ]:
# --- Graphique 5 : Problèmes résolus vs seuil ---
fig5 = px.line(df_sim, x='threshold', y='pct_problems_solved', color='scope', markers=True,
    title='% de cas problématiques résolus vs Seuil',
    labels={'threshold': 'Seuil (min)', 'pct_problems_solved': '% Résolus'},
    color_discrete_map={'all': '#636EFA', 'connect': '#EF553B'})
for t in [60, 120, 180]:
    fig5.add_vline(x=t, line_dash='dot', line_color='grey', annotation_text=str(t)+' min')
fig5.update_layout(template='plotly_white', yaxis_ticksuffix='%')
fig5.show()

In [ ]:
# --- Graphique 6 : Chiffre d'affaires affecté vs seuil ---
fig6 = px.line(df_sim, x='threshold', y='pct_rentals_affected', color='scope', markers=True,
    title='% de locations affectées (Revenue à Risque) vs Seuil',
    labels={'threshold': 'Seuil (min)', 'pct_rentals_affected': '% Affectées'},
    color_discrete_map={'all': '#636EFA', 'connect': '#EF553B'})
for t in [60, 120, 180]:
    fig6.add_vline(x=t, line_dash='dot', line_color='grey', annotation_text=str(t)+' min')
fig6.update_layout(template='plotly_white', yaxis_ticksuffix='%')
fig6.show()

In [ ]:
# --- Graphique 7 : Graphique de compromis (double axe Y) pour scope='all' ---
df_all = df_sim[df_sim['scope'] == 'all']
fig7 = make_subplots(specs=[[{'secondary_y': True}]])
fig7.add_trace(go.Scatter(x=df_all['threshold'], y=df_all['pct_problems_solved'],
    name='% Problèmes Résolus', line=dict(color='#00CC96', width=2), mode='lines+markers'),
    secondary_y=False)
fig7.add_trace(go.Scatter(x=df_all['threshold'], y=df_all['pct_rentals_affected'],
    name='% Revenue à Risque', line=dict(color='#EF553B', width=2), mode='lines+markers'),
    secondary_y=True)
fig7.update_layout(title='Compromis : Problèmes Résolus vs Revenu à Risque (Toutes Voitures)',
    template='plotly_white', xaxis_title='Seuil (min)')
fig7.update_yaxes(title_text='% Problèmes Résolus', secondary_y=False, ticksuffix='%')
fig7.update_yaxes(title_text='% Revenu à Risque',  secondary_y=True,  ticksuffix='%')
fig7.show()

for t in [60, 120, 180]:
    row = df_sim[(df_sim['threshold']==t) & (df_sim['scope']=='all')].iloc[0]
    print(f"Seuil {t}min : {row['pct_problems_solved']:.1f}% résolus, "
          f"{row['pct_rentals_affected']:.1f}% de revenus affectés")

### Recommandation Business

| Seuil | Problèmes Résolus | Risque Revenu | Verdict |
|-------|-------------------|---------------|---------|
| 60 min | ~XX% | ~X% | Meilleur équilibre |
| 120 min | ~XX% | ~X% | Plus de couverture, coût plus élevé |
| 180 min | ~XX% | ~X% | Trop coûteux |

**Recommandation :** Un **seuil de 60 minutes** sur **toutes les voitures** offre le meilleur compromis.

---
## 4. Partie 3 — EDA du Jeu de Données de Tarification

### 4.1 Vérification de la Qualité des Données

In [ ]:
# --- Aperçu du jeu de données de tarification ---
if 'Unnamed: 0' in df_pricing.columns:
    df_pricing = df_pricing.drop(columns=['Unnamed: 0'])
    print('Colonne Unnamed: 0 supprimée')

print(f'Dimensions : {df_pricing.shape}  |  Manquantes : {df_pricing.isnull().sum().sum()}')

# --- Identifier les types de caractéristiques ---
num_features  = ['mileage', 'engine_power']
cat_features  = ['model_key', 'fuel', 'paint_color', 'car_type']
bool_features = ['private_parking_available', 'has_gps', 'has_air_conditioning',
                 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
target = 'rental_price_per_day'

print(f'Numériques : {num_features}')
print(f'Catégorielles : {cat_features}')
print(f'Booléennes : {bool_features}')

### 4.2 Distribution de la Variable Cible

In [ ]:
# --- Graphique 8 : Distribution de la cible ---
fig8 = px.histogram(df_pricing, x='rental_price_per_day', nbins=60,
    title='Distribution du prix de location journalier (Cible)',
    labels={'rental_price_per_day': 'Prix / Jour (EUR)'},
    color_discrete_sequence=['#636EFA'])
med = df_pricing['rental_price_per_day'].median()
avg = df_pricing['rental_price_per_day'].mean()
fig8.add_vline(x=med, line_dash='dash', line_color='red',
               annotation_text='Médiane : ' + str(int(med)) + ' EUR')
fig8.add_vline(x=avg, line_dash='dash', line_color='orange',
               annotation_text='Moyenne : ' + str(int(avg)) + ' EUR')
fig8.update_layout(template='plotly_white')
fig8.show()
print(df_pricing['rental_price_per_day'].describe())

### 4.3 Corrélations & Nuages de Points

In [ ]:
# --- Graphique 9 : Carte de chaleur des corrélations ---
df_corr = df_pricing.copy()
for col in bool_features:
    df_corr[col] = df_corr[col].astype(int)
corr = df_corr[num_features + bool_features + [target]].corr()
fig9 = px.imshow(corr, title='Carte de chaleur des corrélations',
    color_continuous_scale='RdBu', text_auto='.2f', zmin=-1, zmax=1)
fig9.update_layout(template='plotly_white')
fig9.show()

In [ ]:
# --- Graphique 10 : Puissance moteur vs Prix ---
fig10 = px.scatter(df_pricing, x='engine_power', y='rental_price_per_day', color='car_type',
    title='Puissance moteur vs Prix de location journalier', opacity=0.6, trendline='ols',
    labels={'engine_power': 'Puissance moteur (ch)', 'rental_price_per_day': 'Prix (EUR/jour)'})
fig10.update_layout(template='plotly_white')
fig10.show()

In [ ]:
# --- Graphique 11 : Kilométrage vs Prix ---
fig11 = px.scatter(df_pricing, x='mileage', y='rental_price_per_day', color='fuel',
    title='Kilométrage vs Prix de location journalier', opacity=0.5,
    labels={'mileage': 'Kilométrage (km)', 'rental_price_per_day': 'Prix (EUR/jour)'})
fig11.update_layout(template='plotly_white')
fig11.show()

### 4.4 Caractéristiques Catégorielles : Prix par Catégorie

In [ ]:
# --- Graphique 12 : Prix par type de voiture ---
order_ct = (df_pricing.groupby('car_type')['rental_price_per_day']
            .median().sort_values(ascending=False).index.tolist())
fig12 = px.box(df_pricing, x='car_type', y='rental_price_per_day', color='car_type',
    title='Prix de location journalier par type de voiture', category_orders={'car_type': order_ct})
fig12.update_layout(template='plotly_white', showlegend=False)
fig12.show()

In [ ]:
# --- Graphique 13 : Prix par type de carburant ---
order_fuel = (df_pricing.groupby('fuel')['rental_price_per_day']
              .median().sort_values(ascending=False).index.tolist())
fig13 = px.box(df_pricing, x='fuel', y='rental_price_per_day', color='fuel',
    title='Prix de location journalier par type de carburant', category_orders={'fuel': order_fuel})
fig13.update_layout(template='plotly_white', showlegend=False)
fig13.show()

In [ ]:
# --- Graphique 14 : Prix médian par marque ---
brand_med = (df_pricing.groupby('model_key')['rental_price_per_day']
             .median().sort_values(ascending=False).reset_index())
fig14 = px.bar(brand_med, x='model_key', y='rental_price_per_day', color='rental_price_per_day',
    title='Prix de location journalier médian par marque', color_continuous_scale='Viridis')
fig14.update_layout(template='plotly_white')
fig14.show()

In [ ]:
# --- Graphique 15 : Impact des caractéristiques booléennes sur le prix ---
bool_impact = []
for feat in bool_features:
    w  = df_pricing[df_pricing[feat] == True]['rental_price_per_day'].mean()
    wo = df_pricing[df_pricing[feat] == False]['rental_price_per_day'].mean()
    bool_impact.append({'feature': feat, 'delta': w - wo})
df_bool = pd.DataFrame(bool_impact).sort_values('delta', ascending=False)
fig15 = px.bar(df_bool, x='feature', y='delta', color='delta',
    title='Surprime de prix des caractéristiques booléennes (EUR/jour)',
    color_continuous_scale='RdYlGn', color_continuous_midpoint=0)
fig15.add_hline(y=0, line_color='black')
fig15.update_layout(template='plotly_white')
fig15.show()
print(df_bool.to_string(index=False))

---
## 5. Partie 4 — Feature Engineering & Pipeline de Prétraitement

- **Numériques** (`mileage`, `engine_power`) : `StandardScaler`
- **Catégorielles** (`model_key`, `fuel`, `paint_color`, `car_type`) : `OneHotEncoder`
- **Booléennes** : passage direct (déjà en 0/1)

In [ ]:
# --- Préparer les caractéristiques et la cible ---
X = df_pricing[num_features + cat_features + bool_features].copy()
y = df_pricing[target]

# Convertir les colonnes booléennes en entiers
for col in bool_features:
    X[col] = X[col].astype(int)

# --- Division train/test (80/20) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')

In [ ]:
# --- Construire le pipeline de prétraitement (ColumnTransformer) ---
numeric_transformer     = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combiner tous les transformateurs
preprocessor = ColumnTransformer(transformers=[
    ('num',  numeric_transformer,     num_features),
    ('cat',  categorical_transformer, cat_features),
    ('bool', 'passthrough',           bool_features),  # Booléen : aucune transformation nécessaire
])
print('Pipeline de prétraitement construit : StandardScaler | OneHotEncoder | passthrough')

---
## 6. Partie 5 — Modélisation de Machine Learning (Suivi MLflow)

**Stratégie :**
1. **Référence (Baseline)** — `LinearRegression` : benchmark interprétable
2. **Modèle principal** — `RandomForestRegressor` + GridSearchCV : robuste, non-linéaire, Importance des Caractéristiques
3. **Alternative** — `GradientBoostingRegressor` : comparaison de boosting

Toutes les exécutions sont suivies via **MLflow**.

In [ ]:
# --- Configurer MLflow ---
mlflow_dir = os.path.abspath('../mlflow/mlruns')
os.makedirs(mlflow_dir, exist_ok=True)
mlflow.set_tracking_uri('file://' + mlflow_dir)
mlflow.set_experiment('getaround_pricing')
print('URI de suivi MLflow :', mlflow.get_tracking_uri())

# --- Aide : entraîner, évaluer, enregistrer ---
def evaluate_model(name, pipeline, params=None):
    with mlflow.start_run(run_name=name):
        if params:
            mlflow.log_params(params)
        mlflow.log_param('model_name', name)
        pipeline.fit(X_train, y_train)
        y_pred  = pipeline.predict(X_test)
        y_tr_pr = pipeline.predict(X_train)
        metrics = {
            'train_r2' : r2_score(y_train, y_tr_pr),
            'test_r2'  : r2_score(y_test, y_pred),
            'test_rmse': float(np.sqrt(mean_squared_error(y_test, y_pred))),
            'test_mae' : float(mean_absolute_error(y_test, y_pred)),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, 'model')
    print(name, metrics)
    return pipeline, y_pred, metrics

In [ ]:
# MODÈLE 1 : Baseline — Régression Linéaire
pipe_lr = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', LinearRegression())])
lr_model, lr_preds, lr_metrics = evaluate_model('LinearRegression_Baseline', pipe_lr)

In [ ]:
# MODÈLE 2 : Random Forest (GridSearchCV)
pipe_rf = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', RandomForestRegressor(random_state=42))])
param_grid = {
    'model__n_estimators'     : [100, 200],
    'model__max_depth'        : [None, 20, 30],
    'model__min_samples_split': [2, 5],
}
print('Exécution de GridSearchCV...')
grid_rf = GridSearchCV(pipe_rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print('Meilleurs paramètres :', grid_rf.best_params_)
print('Meilleur R2 par CV :', round(grid_rf.best_score_, 4))

best_rf_params = {k.replace('model__', ''): v for k, v in grid_rf.best_params_.items()}
best_rf_params['random_state'] = 42
pipe_rf_best = Pipeline(steps=[('preprocessor', preprocessor),
                                ('model', RandomForestRegressor(**best_rf_params))])
rf_model, rf_preds, rf_metrics = evaluate_model('RandomForest_GridSearch', pipe_rf_best, best_rf_params)

In [ ]:
# MODÈLE 3 : Gradient Boosting
gb_params = dict(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
pipe_gb = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', GradientBoostingRegressor(**gb_params))])
gb_model, gb_preds, gb_metrics = evaluate_model('GradientBoosting', pipe_gb, gb_params)

In [ ]:
# --- Résumé de la comparaison des modèles ---
comparison = pd.DataFrame({
    'Modèle'   : ['Régression Linéaire', 'Random Forest (meilleur)', 'Gradient Boosting'],
    'R2 Train' : [lr_metrics['train_r2'], rf_metrics['train_r2'], gb_metrics['train_r2']],
    'R2 Test'  : [lr_metrics['test_r2'],  rf_metrics['test_r2'],  gb_metrics['test_r2']],
    'RMSE EUR' : [lr_metrics['test_rmse'],rf_metrics['test_rmse'],gb_metrics['test_rmse']],
    'MAE EUR'  : [lr_metrics['test_mae'], rf_metrics['test_mae'], gb_metrics['test_mae']],
})
print(comparison.to_string(index=False))

---
## 7. Partie 6 — Évaluation du Modèle & Importance des Caractéristiques

In [ ]:
# --- Graphique 16 : Prédit vs Réel ---
pred_df = pd.DataFrame({'Réel': y_test, 'Prédit': rf_preds})
pred_df['Erreur'] = abs(pred_df['Prédit'] - pred_df['Réel'])
fig16 = px.scatter(pred_df, x='Réel', y='Prédit', color='Erreur',
    title='Prédit vs Réel — RandomForest',
    color_continuous_scale='RdYlGn_r', opacity=0.7)
mn = min(pred_df['Réel'].min(), pred_df['Prédit'].min())
mx = max(pred_df['Réel'].max(), pred_df['Prédit'].max())
fig16.add_trace(go.Scatter(x=[mn,mx], y=[mn,mx], mode='lines',
    name='Prédiction parfaite', line=dict(color='black', dash='dash')))
fig16.update_layout(template='plotly_white')
fig16.show()

In [ ]:
# --- Graphique 17 : Importance des Caractéristiques ---
# Obtenir les noms des caractéristiques après prétraitement
cat_names = rf_model.named_steps['preprocessor']\
    .transformers_[1][1]['onehot'].get_feature_names_out(cat_features).tolist()
feat_names = num_features + cat_names + bool_features
importances = rf_model.named_steps['model'].feature_importances_

fi_df = pd.DataFrame({'Caractéristique': feat_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=True).tail(20)
fig17 = px.bar(fi_df, x='Importance', y='Caractéristique', orientation='h',
    title='Importance des Caractéristiques — RandomForest (Top 20)',
    color='Importance', color_continuous_scale='Viridis')
fig17.update_layout(template='plotly_white', height=600)
fig17.show()

In [ ]:
# --- Sauvegarder le meilleur modèle + infos caractéristiques pour l'API ---
os.makedirs('../api', exist_ok=True)
joblib.dump(rf_model, '../api/model.pkl')
feature_info = {
    'num_features' : num_features,
    'cat_features' : cat_features,
    'bool_features': bool_features,
}
with open('../api/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)
print('Modèle sauvegardé  -> ../api/model.pkl')
print('Caratéristiques sauvegardées -> ../api/feature_info.json')

In [ ]:
# --- Résumé final ---
print('='*55)
print('RÉSULTATS FINAUX — Modèle de Tarification GetAround')
print('='*55)
print(f'Meilleur modèle : RandomForestRegressor')
print(f'R2 Test         : {rf_metrics["test_r2"]:.4f}')
print(f'RMSE Test       : {rf_metrics["test_rmse"]:.2f} EUR/jour')
print(f'MAE Test        : {rf_metrics["test_mae"]:.2f} EUR/jour')
fi_top = pd.DataFrame({'Caractéristique': feat_names, 'Importance': importances})\
           .sort_values('Importance', ascending=False)
print('Top 5 Caractéristiques :')
print(fi_top.head(5).to_string(index=False))